# LC 684 — Redundant Connection
**Day 73 · Union-Find · Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Process edges one by one. The first
edge whose two endpoints are <em>already in the same component</em>
(find(u) == find(v)) is the redundant edge — it would create a
cycle. Return it immediately.
</div>

## Official Problem Statement

You are given a graph that started as a tree with `n` nodes labelled
`1` to `n`, with one **additional edge** added. The added edge has
two different vertices chosen from `1` to `n` and was not an edge
that already existed.

The graph is represented as an array `edges` where
`edges[i] = [ai, bi]` indicates an undirected edge between nodes
`ai` and `bi`.

Return an edge that can be removed so that the resulting graph is a
tree of `n` nodes. If there are multiple answers, return the edge
that **occurs last** in the input.

**Constraints:**
- `n == edges.length`
- `3 <= n <= 1000`
- `edges[i]` has no self-loops or repeated edges

## What This Is Actually Asking

A tree with `n` nodes has exactly `n-1` edges and no cycles.
We have `n` edges total, so exactly **one extra edge** was added,
creating exactly one cycle.

We need to find which edge is the "guilty" one — the one that,
if removed, leaves a valid tree.

Because there may be multiple edges in the cycle, the problem
says: return the one that appears **last** in the `edges` array.
Processing edges in order and detecting the first one that would
form a cycle naturally gives us the last-appearing redundant edge.

## Walk Through an Example by Hand

```
edges = [[1,2], [1,3], [2,3]]
n = 3
```

**Initial:** parent = [_, 1, 2, 3]  (1-indexed; index 0 unused)
             rank   = [_, 0, 0, 0]

**Edge [1,2]:** find(1)=1, find(2)=2 — different → union(1,2)
```
  parent = [_, 1, 1, 3]   rank = [_, 1, 0, 0]
```

**Edge [1,3]:** find(1)=1, find(3)=3 — different → union(1,3)
```
  parent = [_, 1, 1, 1]   rank = [_, 1, 0, 0]
```

**Edge [2,3]:** find(2)=find(1)=1, find(3)=1
```
  Same root! → CYCLE DETECTED → return [2,3] ✓
```

## The Picture

Union-Find parent array evolution — edges [[1,2],[1,3],[2,3]]

```
Nodes: 1, 2, 3  (1-indexed)

Initial:
  idx:    1   2   3
  parent: 1   2   3   ← each node is its own root
  rank:   0   0   0

Process [1,2]: find(1)=1, find(2)=2 → DIFFERENT → union
  same rank → attach 2 under 1, bump rank[1]
  idx:    1   2   3
  parent: 1   1   3
  rank:   1   0   0

  Tree so far:  1 — 2

Process [1,3]: find(1)=1, find(3)=3 → DIFFERENT → union
  rank[1]=1 > rank[3]=0 → attach 3 under 1
  idx:    1   2   3
  parent: 1   1   1
  rank:   1   0   0

  Tree so far:  2 — 1 — 3

Process [2,3]: find(2)→parent[2]=1→return 1
               find(3)→parent[3]=1→return 1
  SAME ROOT (1) → CYCLE!
  ╔══════════════════════════════╗
  ║  Return [2, 3] immediately  ║
  ╚══════════════════════════════╝

Path compression example (deeper tree):
  Before:  4 → 3 → 1    After find(4):  4 → 1  (shortcut!)
  parent[4] collapses to point directly at root 1
```

## When To Use This Pattern

Use Union-Find cycle detection when:
- You add edges **incrementally** and need to detect the moment a
  cycle forms
- The graph is **undirected** (directed cycle detection needs DFS)
- You need the **specific edge** that creates the cycle

**Signals in the problem:**
- "Tree + one extra edge" description
- "Remove an edge to make it a tree"
- "Detect the redundant / duplicate connection"

**Key invariant:** Before a cycle-forming edge is processed,
the graph is always a valid forest. The first edge where
`find(u) == find(v)` is the cycle edge.

## The Approach

1. **Initialise** `parent[i] = i` and `rank[i] = 0` for
   i in range(n+1)  (1-indexed, index 0 unused).

2. **find(x)** with path compression:
   - If `parent[x] != x`: `parent[x] = find(parent[x])`
   - Return `parent[x]`

3. **union(a, b)** with union by rank:
   - `ra, rb = find(a), find(b)`
   - If `ra == rb`: **return False** (cycle detected!)
   - If `rank[ra] < rank[rb]`: swap ra, rb
   - `parent[rb] = ra`
   - If `rank[ra] == rank[rb]`: `rank[ra] += 1`
   - Return True

4. **Iterate** over `edges`:
   - For each `[u, v]`: if `not union(u, v)`: return `[u, v]`

5. Return `[]`  (should never reach here given valid input).

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """
    Run test cases for findRedundantConnection.
    Prints PASSED / FAILED and a summary line.
    """
    tests = [
        # (edges, expected)
        ([[1,2],[1,3],[2,3]],           [2,3]),
        ([[1,2],[2,3],[3,4],[1,4],[1,5]], [1,4]),
        ([[1,2],[2,3],[3,1]],           [3,1]),
        (
            [[1,4],[3,4],[1,3],[1,2],[4,5]],
            [1,3]
        ),
        (
            [[2,7],[7,8],[3,6],[2,5],[6,8],
             [2,8],[1,3],[2,4],[1,9],[1,10]],
            [2,8]
        ),
    ]
    passed = 0
    for i, (edges, expected) in enumerate(tests):
        result = func(edges)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"Test {i+1}: {status} "
            f"(got {result}, expected {expected})"
        )
    total = len(tests)
    print(f"\nResult: {passed}/{total} tests passed.")

In [ ]:
def findRedundantConnection(
    edges: List[List[int]]
) -> List[int]:
    """
    LC 684 — Redundant Connection.

    Process edges one by one. Return the first edge that
    would form a cycle (find(u) == find(v)).

    Args:
        edges: List of [u, v] undirected edges.
            Nodes are 1-indexed, 1 <= u, v <= n.

    Returns:
        The redundant edge [u, v] that appears last
        in the input and can be removed to form a tree.

    Examples:
        >>> findRedundantConnection([[1,2],[1,3],[2,3]])
        [2, 3]
        >>> findRedundantConnection(
        ...     [[1,2],[2,3],[3,4],[1,4],[1,5]])
        [1, 4]

    Plan:
        1. parent[i]=i, rank[i]=0 for i in 0..n (1-indexed)
        2. find(x) with path compression
        3. union(a,b) returns False if cycle detected
        4. Iterate edges; return edge on cycle detection
    """
    n = len(edges)
    parent = list(range(n + 1))  # 1-indexed
    rank = [0] * (n + 1)

    # Debug: show initial parent slice
    print(f"[DEBUG] n={n}, parent[1..n]={parent[1:n+1]}")

    def find(x: int) -> int:
        """Path-compressed find."""
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(a: int, b: int) -> bool:
        """
        Union by rank. Returns False if a and b share
        the same root (cycle detected).
        """
        ra, rb = find(a), find(b)
        if ra == rb:
            print(f"  [DEBUG] CYCLE on ({a},{b}), root={ra}")
            return False
        if rank[ra] < rank[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        if rank[ra] == rank[rb]:
            rank[ra] += 1
        print(
            f"  [DEBUG] union({a},{b}) ok "
            f"→ parent[1..n]={parent[1:n+1]}"
        )
        return True

    pass  # TODO: iterate over edges, return cycle edge

    return []  # should not reach here

In [ ]:
# Uncomment and run when solution is ready
# test_harness(findRedundantConnection)

## Complexity

| | Value |
|---|---|
| **Time** | O(n·α(n)) — iterate through n edges; each find/union is near O(1) with path compression + union by rank |
| **Space** | O(n) — `parent` and `rank` arrays of size n+1 |

**Note:** α(n) is the inverse Ackermann function — it grows
so slowly it is effectively constant for all practical n.
This is essentially O(n) time and O(n) space.

## Real World Connection

**Detecting circular dependencies in build systems:**
In tools like Bazel or Make, build targets have dependency edges.
A circular dependency (A depends on B, B depends on A) would
cause an infinite build loop.

When a developer adds a new dependency edge, the build system
runs exactly this algorithm: Union-Find on all known dependency
edges processed in the order they were declared. The first edge
where `find(source) == find(target)` is flagged immediately as
the redundant / circular dependency, and the build fails with
a precise error pointing to that exact declaration.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra